Seurat data to Python 

In [ ]:
import scanpy as sc

In [ ]:
#small data set -> did not work for this, was missing genes
#adata = sc.read_h5ad(r'C:\Users\jasmi\OneDrive\Desktop\AlzheimersResearch\seurat_object_fire_wt_01.h5ad')


In [ ]:
#larger data set
adata = sc.read_h5ad(r"C:\Users\jasmi\OneDrive\Desktop\AlzheimersResearch\seurat_object_fire_wt.h5ad")


checking clusters

In [ ]:
# 1. List all available metadata columns
print("Available columns in metadata:")
print(adata.obs.columns.tolist())

# 2. Look at the first few rows to see what the data actually looks like
print("\nFirst 5 rows of metadata:")
display(adata.obs.head())

In [ ]:
column_to_check = 'mmc_class_name' 
if column_to_check in adata.obs.columns:
    print(f"\nCell counts for '{column_to_check}':")
    print(adata.obs[column_to_check].value_counts())

Cut off at Immune class

In [ ]:
# # 1. Ensure the column is categorical (this is very fast)
# adata.obs['mmc_class_name'] = adata.obs['mmc_class_name'].astype('category')

# # 2. Define your keep list
# keep_classes = [
#     '01 IT-ET Glut', '02 NP-CT-L6b Glut', '07 CTX-MGE GABA', 
#     '31 OPC-Oligo', '06 CTX-CGE GABA', '33 Vascular', 
#     '30 Astro-Epen', '09 CNU-LGE GABA', '34 Immune'
# ]

# # 3. Filter WITHOUT .copy() first to see if it's faster
# # This creates a "View" (a pointer) rather than a new object
# adata_subset = adata[adata.obs['mmc_class_name'].isin(keep_classes)]

Steady- bold textstate Ligand-Receptor inference

In [ ]:
# import liana
import liana as li
# needed for visualization and toy data
import scanpy as sc

I have changed the data from toy data to the converted seurat data and mofidied the below graph to group by the column "seurat_clusters" instead
and now: broad_cell_type

In [ ]:
sc.pl.umap(adata, color='mmc_class_name', title='', frameon=False)

INFO

We have the overall gene expression for each cell.

*-> What to do with that data?*
One thing to do is to **group the cells together by gene expression** (microglia have different gene expressions than neurons, astrocytes, etc., as different proteins have different structures)
*-> Can we group cells together by similar gene expression patterns?*
Yes. The above graph does something similar.

**Clustering** is working out which cells are close together in a space in order to find genes with similar expression patterns. After clustering, each cell has a label with which cluster it is in, and the cluster labels identify which cells have similar expression patterns. This data set is very high dimensional.

**The above graph is a visualization of clustering.**

It's very hard to vsiualize single-cell data in 2d. UMAP is an algorithm that takes this form of data and tries to represent it in 2d such that cells with similar expression patterns are close together and cells with different expression patterns are far apart.

LIANA typically uses log1p-transformed counts matrix, which is...

To look at what's going on in cells, one can look at changes in protein expression, which is difficult so it's easier to look at levels of RNA as they translate into proteins.

microglia changes behavior by making proteins (by making RNA). By looking at RNA sequencing data, we can look at which genes are being expressed.

The data we are using in the actual project (not the toy data in this tutorial) is from the cells we want to measure the expression in, with the cells put in a machine that extracts all of the RNA and attaches a cell indictor barcode to the RNA molecules. You can then see the distribution of RNA molecules per cell (single cell) and in the brain in general. The single cell data is what we are using for the project (distribution of RNA on a single-cell level).

Our data will be a big matrix of A x B, A being cells (each column a cell), and B being genes (each row a gene), and the matrix shows how much of a certain gene is in a particular cell (which one is r/c is arbitrary right now).

In [ ]:
adata.raw.X

In [ ]:
li.mt.show_methods()

In [ ]:
# import liana's rank_aggregate
from liana.mt import rank_aggregate

In [ ]:
rank_aggregate.__call__

In [ ]:
rank_aggregate.describe()

In [ ]:
# import all individual methods
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean

switched resource to mouseresource as toy data was human genes

In [ ]:
# run cellphonedb
cellphonedb(adata,
            groupby='mmc_class_name',
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name='mouseconsensus', 
            expr_prop=0.1,
            verbose=True, key_added='cpdb_res')

In [ ]:
# by default, liana's output is saved in place:
adata.uns['cpdb_res'].head()

Dotplot

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# li.pl.dotplot(adata = adata,
#               colour='lr_means',
#               size='cellphone_pvals',
#               inverse_size=True, # we inverse sign since we want small p-values to have large sizes
#               source_labels=['CD34+', 'CD56+ NK', 'CD14+ Monocyte'],
#               target_labels=['CD34+', 'CD56+ NK'],
#               figure_size=(8, 7),
#               # finally, since cpdbv2 suggests using a filter to FPs
#               # we filter the pvals column to <= 0.05
#               filter_fun=lambda x: x['cellphone_pvals'] <= 0.05,
#               uns_key='cpdb_res' # uns_key to use, default is 'liana_res'
#              )

In [ ]:
# This will show you exactly what columns were calculated
print(adata.uns['cpdb_res'].columns)

In [ ]:
top_df = adata.uns['cpdb_res'].sort_values('lr_means', ascending=False).head(20)

In [ ]:
li.pl.dotplot(liana_res = top_df,
              colour = 'lr_means',
              size = 'cellphone_pvals',
              inverse_size = True,
              figure_size = (17, 20)
             )


In [ ]:
# Run this once to tell LIANA that 'cpdb_res' is the main result
# adata.uns['liana_res'] = adata.uns['cpdb_res']

original dot plot code and modifications of it do not work

In [ ]:
# li.pl.dotplot(adata = adata,
#               liana_res = adata.uns['cpdb_res'], 
#               colour = 'lr_means',
#               size = 'cellphone_pvals',
#               inverse_size = True,
#               # Explicitly define these to prevent the 'None' error
#               ligand_complex = 'ligand_complex',
#               receptor_complex = 'receptor_complex',
#               top_n = 20, 
#               figure_size = (10, 8)
#              )

In [ ]:
my_plot = li.pl.tileplot(adata = adata,
                         # NOTE: fill & label need to exist for both
                         # ligand_ and receptor_ columns
                         fill='means',
                         label='props',
                         label_fun=lambda x: f'{x:.2f}',
                         top_n=10,
                         orderby='cellphone_pvals',
                         orderby_ascending=True,
                        #  source_labels=['CD34+', 'CD56+ NK', 'CD14+ Monocyte'],
                        #  target_labels=['CD34+', 'CD56+ NK'],
                         uns_key='cpdb_res', # NOTE: default is 'liana_res'
                         source_title='Ligand',
                         target_title='Receptor',
                         figure_size=(15, 7),
                         label_size=7
                         )
my_plot

INFO

Ligand in this case acts as a communication molecule that is emitted, which is detected by the receptors.
Look at ligand-producing gene expression and same for receptor, you could see (for example) that certain types of cells might express certain ligands and some might express certain receptors. If we know cell type 1 expresses a gene associated with a particular ligand and cell type 2 with a particular receptor, and that ligand-receptor pair is one we know to communicate, then we know that those cells might be communicating with one another.

We want to look at communication changes in Alzheimer's mice eventually.

In [ ]:
# Run rank_aggregate
li.mt.rank_aggregate(adata,
                     groupby='mmc_class_name',
                     resource_name='mouseconsensus',
                     expr_prop=0.1,
                     verbose=True)

In [ ]:
rank_aggregate.describe()

In [ ]:
li.pl.dotplot(adata = adata,

                        uns_key = 'cpdb_res',

                        colour = 'lr_means',

                        size = 'cellphone_pvals',

                        inverse_size = True, # High significance (low p-val) = Large dot

                       

                        # FIX: Tell LIANA how to find the "top" 20

                        orderby = 'lr_means',

                        orderby_ascending = False, # Largest means at the top

                       

                        filter_fun = lambda x: x['cellphone_pvals'] <= 0.05,

                        top_n = 20,

                        figure_size = (12, 25)

                       )



# To display in a notebook, you usually just call the object

my_plot

In [ ]:
# my_plot = li.pl.dotplot(adata = adata,
#                         colour='magnitude_rank',
#                         inverse_colour=True,
#                         size='specificity_rank',
#                         inverse_size=True,
#                         # source_labels=['CD34+', 'CD56+ NK', 'CD14+ Monocyte'],
#                         # target_labels=['CD34+', 'CD56+ NK'],
#                         filter_fun=lambda x: x['specificity_rank'] <= 0.01,
#                         figure_size=(20,10),
#                         top_n=30
#                        )
# my_plot

In [ ]:
my_plot = li.pl.dotplot(adata = adata,
                        uns_key = 'cpdb_res',
                        colour = 'lr_means',
                        size = 'cellphone_pvals',
                        inverse_size = True, # High significance (low p-val) = Large dot
                        
                        # FIX: Tell LIANA how to find the "top" 20
                        orderby = 'lr_means', 
                        orderby_ascending = False, # Largest means at the top
                        
                        filter_fun = lambda x: x['cellphone_pvals'] <= 0.05,
                        top_n = 20,
                        figure_size = (25, 25)
                       )

# To display in a notebook, you usually just call the object
my_plot


In [ ]:
# we import plotnine
import plotnine as p9

In [ ]:
# # (my_plot +
# #  # change theme
# #  p9.theme_dark() +
# #  # modify theme
# #  p9.theme(
# #      # adjust facet size
# #      strip_text=p9.element_text(size=11),
# #      figure_size = (25, 20)
# #  )
# # )
# p9.theme(
#     # Rotate 90 degrees and align the end of the text to the axis
#     axis_text_x=p9.element_text(rotation=90, va='center', ha='right', size=10),
#     figure_size=(25, 20)
# )
# optimized_plot = (
#     my_plot +
#     p9.geom_point(size=0.1, alpha=0.3) +
#     p9.theme_dark() +
#     p9.theme(
#         # 1. Rotate fully vertical
#         axis_text_x=p9.element_text(rotation=90, hjust=1, size=9), 
#         # 2. Increase spacing between the sub-plots (facets)
#         panel_spacing_x=0.8, 
#         # 3. Increase font of the facet headers (top labels)
#         strip_text=p9.element_text(size=12),
#         # 4. Large canvas size
#         figure_size=(25, 20)
#     )
# )

# optimized_plot
# (my_plot +
#  # change theme
#  p9.theme_dark() +
#  # modify theme
#  p9.theme(
#      # adjust facet size
#      strip_text=p9.element_text(size=11),
#      figure_size = (25, 20)
#  )
# )
p9.theme(
    # Rotate 90 degrees and align the end of the text to the axis
    axis_text_x=p9.element_text(rotation=90, va='center', ha='right', size=10),
    figure_size=(25, 20)
)
optimized_plot = (
    my_plot +
    p9.geom_point(size=0.1, alpha=0.3) +
    p9.theme_dark() +
    p9.theme(
        # 1. Rotate fully vertical
        axis_text_x=p9.element_text(rotation=90, hjust=1, size=9), 
        # 2. Increase spacing between the sub-plots (facets)
        panel_spacing_x=0.8, 
        # 3. Increase font of the facet headers (top labels)
        strip_text=p9.element_text(size=12),
        # 4. Large canvas size
        figure_size=(25, 20)
    )
)

optimized_plot


In [ ]:
li.pl.circle_plot(adata,
                  groupby='mmc_class_name',
                  score_key='magnitude_rank',
                  inverse_score=True,
                #   source_labels='CD34+',
                  filter_fun=lambda x: x['specificity_rank'] <= 0.05,
                  pivot_mode='counts', # NOTE: this will simply count the interactions, 'mean' is also available
                  figure_size=(10, 10),
                  )

In [ ]:
methods = [logfc, geometric_mean]
new_rank_aggregate = li.mt.AggregateClass(li.mt.aggregate_meta, methods=methods)

In [ ]:
new_rank_aggregate(adata,
                   groupby='mmc_class_name',
                   expr_prop=0.1,
                   verbose=True,
                   resource_name='mouseconsensus',
                   # Note that with this option, we don't perform permutations
                   # and hence we exclude the p-value for geometric_mean, as well as specificity_rank
                   n_perms=None,
                   use_raw=True,
                   )

In [ ]:
adata.uns['liana_res'].head()